In [38]:
import csv
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path


import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights

In [39]:
if torch.cuda.is_available():
    device = torch.device('cuda')
    print("Running on GPU")
else:
    device = torch.device('cpu')
    print("Running on CPU")

Running on GPU


In [40]:
# Config for dataset
DATASET = 'MIT-dataset'  # 'MIT-dataset' or 'salicon-t', you can insert another dataset but it must have the structure images and maps

num_examples = 4
batch_size = 4

In [41]:
#for each run, all three folds are evaluated
#set the run by deleting the hashtag corresponding to it
RUNS = {
    #'SS_scratch': 'SS',
    # 'SS_pretrained_finetuned': 'SS',
    # 'SS_pretrained_frozen': 'SS',
    # 'ML_scratch': 'ML',
     'ML_pretrained_finetuned': 'ML',
    #'ML_pretrained_frozen': 'ML',
}
folds = (1, 2, 3)
checkpoints = {f'{run}_fold{fold}': (architecture, Path('training_runs') / run / f'fold_{fold}' / 'best_model.pth')
    for run, architecture in RUNS.items() for fold in folds}

output_dir = Path('testing_results') / DATASET
output_dir.mkdir(parents=True, exist_ok=True)

## Load the models


#### SSResNet18

In [42]:
class SSEncoder(nn.Module):

    def __init__(self, pretrained=False, frozen=False):
        super().__init__()
        if pretrained:
            weights = ResNet18_Weights.IMAGENET1K_V1
        else:
            weights = None
        resnet = resnet18(weights=weights)
        self.conv1 = resnet.conv1
        self.bn1 = resnet.bn1
        self.relu = resnet.relu
        self.maxpool = resnet.maxpool
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4
        self.layer4[0].conv1.stride = (1, 1)
        self.layer4[0].downsample[0].stride = (1, 1)
        self.frozen = frozen
        if self.frozen:
            self.freeze()

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return x

    def freeze(self):
        """Freeze all encoder parameters and set encoder in eval mode"""
        self.frozen = True
        for p in self.parameters():
            p.requires_grad = False
        self.eval()

    def unfreeze(self):
        """unfreeze encoder parameters"""
        self.frozen = False
        for p in self.parameters():
            p.requires_grad = True
        self.train()

    def train(self, mode=True):
        """Keep the encoder in eval mode when is frozen (it stops batchnorm layers)"""
        super().train(mode)
        if self.frozen:
            super().train(False)
        return self

class SSDecoder(nn.Module):

    def __init__(self):
        super().__init__()
        self.up_layer1 = nn.Sequential(nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True))
        self.up_layer2 = nn.Sequential(nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True))
        self.up_layer3 = nn.Sequential(nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True))
        self.up_layer4 = nn.Sequential(nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True))
        self.classifier = nn.Conv2d(in_channels=32, out_channels=1, kernel_size=1)

    def forward(self, x):
        x = self.up_layer1(x)
        x = self.up_layer2(x)
        x = self.up_layer3(x)
        x = self.up_layer4(x)
        return self.classifier(x)

class SSResNet18(nn.Module):

    def __init__(self, pretrained=False, frozen=False):
        super().__init__()
        self.encoder = SSEncoder(pretrained=pretrained, frozen=frozen)
        self.decoder = SSDecoder()

    def forward(self, x):
        if self.encoder.frozen:
            with torch.no_grad():
                features = self.encoder(x)
        else:
            features = self.encoder(x)
        logits = self.decoder(features)
        return logits

    def freeze_encoder(self):
        self.encoder.freeze()

    def unfreeze_encoder(self):
        self.encoder.unfreeze()


#### MLResNet18

In [43]:
class MLEncoder(nn.Module):

    def __init__(self, pretrained=False, frozen=False):
        super().__init__()
        if pretrained:
            weights = ResNet18_Weights.IMAGENET1K_V1
        else:
            weights = None
        resnet = resnet18(weights=weights)
        self.conv1 = resnet.conv1
        self.bn1 = resnet.bn1
        self.relu = resnet.relu
        self.maxpool = resnet.maxpool
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4
        self.layer4[0].conv1.stride = (1, 1)
        self.layer4[0].downsample[0].stride = (1, 1)
        self.frozen = frozen
        if self.frozen:
            self.freeze()

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        feature1 = self.layer1(x)
        feature2 = self.layer2(feature1)
        layer3_output = self.layer3(feature2)
        feature3 = self.layer4(layer3_output)
        return (feature1, feature2, feature3)

    def freeze(self):
        """Freeze all encoder parameters and set encoder in eval mode"""
        self.frozen = True
        for p in self.parameters():
            p.requires_grad = False
        self.eval()

    def unfreeze(self):
        """Unfreeze encoder parameters"""
        self.frozen = False
        for p in self.parameters():
            p.requires_grad = True
        self.train()

    def train(self, mode=True):
        """Keep encoder in eval mode when frozen."""
        super().train(mode)
        if self.frozen:
            super().train(False)
        return self

class MLDecoder(nn.Module):

    def __init__(self):
        super().__init__()
        self.projection1 = nn.Conv2d(64, 256, kernel_size=1)
        self.projection2 = nn.Conv2d(128, 512, kernel_size=1)
        self.projection3 = nn.Conv2d(512, 512, kernel_size=1)
        self.decoder1_up_layer1 = nn.Sequential(nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1), nn.ReLU(inplace=True))
        self.decoder1_up_layer2 = nn.Sequential(nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1), nn.ReLU(inplace=True))
        self.classifier1 = nn.Conv2d(64, 1, kernel_size=1)
        self.decoder2_up_layer1 = nn.Sequential(nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1), nn.ReLU(inplace=True))
        self.decoder2_up_layer2 = nn.Sequential(nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1), nn.ReLU(inplace=True))
        self.decoder2_up_layer3 = nn.Sequential(nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1), nn.ReLU(inplace=True))
        self.classifier2 = nn.Conv2d(64, 1, kernel_size=1)
        self.decoder3_up_layer1 = nn.Sequential(nn.ConvTranspose2d(512, 512, kernel_size=4, stride=2, padding=1), nn.ReLU(inplace=True))
        self.decoder3_up_layer2 = nn.Sequential(nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1), nn.ReLU(inplace=True))
        self.decoder3_up_layer3 = nn.Sequential(nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1), nn.ReLU(inplace=True))
        self.decoder3_up_layer4 = nn.Sequential(nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1), nn.ReLU(inplace=True))
        self.classifier3 = nn.Conv2d(64, 1, kernel_size=1)
        self.fusion = nn.Conv2d(3, 1, kernel_size=1)

    def forward(self, features):
        feature1, feature2, feature3 = features
        feature1 = self.projection1(feature1)
        feature2 = self.projection2(feature2)
        feature3 = self.projection3(feature3)
        x1 = self.decoder1_up_layer1(feature1)
        x1 = self.decoder1_up_layer2(x1)
        logit1 = self.classifier1(x1)
        x2 = self.decoder2_up_layer1(feature2)
        x2 = self.decoder2_up_layer2(x2)
        x2 = self.decoder2_up_layer3(x2)
        logit2 = self.classifier2(x2)
        x3 = self.decoder3_up_layer1(feature3)
        x3 = self.decoder3_up_layer2(x3)
        x3 = self.decoder3_up_layer3(x3)
        x3 = self.decoder3_up_layer4(x3)
        logit3 = self.classifier3(x3)
        multi_level_logits = torch.cat((logit1, logit2, logit3), dim=1)
        fused_logit = self.fusion(multi_level_logits)
        return {'level1': logit1, 'level2': logit2, 'level3': logit3, 'fused': fused_logit}

class MLResNet18(nn.Module):

    def __init__(self, pretrained=False, frozen=False):
        super().__init__()
        self.encoder = MLEncoder(pretrained=pretrained, frozen=frozen)
        self.decoder = MLDecoder()

    def forward(self, x):
        if self.encoder.frozen:
            with torch.no_grad():
                features = self.encoder(x)
        else:
            features = self.encoder(x)
        outputs = self.decoder(features)
        return outputs

    def freeze_encoder(self):
        self.encoder.freeze()

    def unfreeze_encoder(self):
        self.encoder.unfreeze()


## Load the dataset

In [44]:
#basically the same classes as in training and tuning. I added just a check for image extensions in the path creation, excluding also hidden files and subdirectories if tehere are some

img_transf = transforms.Compose([transforms.Resize((480, 640)), transforms.ToTensor(), transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))])
map_transf = transforms.Compose([transforms.Resize((480, 640)), transforms.ToTensor()])

class create_dataset(Dataset):
    def __init__(self, image_paths, map_paths=None, image_transform=img_transf, map_transform=map_transf):
        self.image_paths = image_paths
        self.map_paths = map_paths
        self.image_transform = image_transform
        self.map_transform = map_transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert('RGB')
        if self.image_transform is not None:
            image = self.image_transform(image)
        if self.map_paths is None:
            return image
        saliency_map = Image.open(self.map_paths[idx]).convert('L')
        if self.map_transform is not None:
            saliency_map = self.map_transform(saliency_map)
        return image, saliency_map


def create_path(path):
    extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
    return sorted(p for p in Path(path).iterdir()
                  if p.is_file() and not p.name.startswith(".")
                  and p.suffix.lower() in extensions)

In [45]:
#### SELECTING THE DATASET ####

#for salicon-t i used the same splitting done with training and tunings, ensuring the same test set for all models
if DATASET == "salicon-t":
    val_images_path = create_path("dataset/images/val/")
    val_maps_path = create_path("dataset/maps/val/")
    
    full_validation_set = create_dataset(val_images_path, val_maps_path)

    rng = np.random.default_rng(42)
    indices = rng.permutation(len(full_validation_set))
    split = int(len(indices) * 0.6)

    test_set = Subset(full_validation_set, indices[split:].tolist())
    test_names = [val_images_path[i].name for i in test_set.indices]

elif DATASET == "MIT-dataset":
    image_paths = create_path("MIT-dataset/images/")
    available_maps = create_path("MIT-dataset/maps/")

    maps_by_name = {path.stem: path for path in available_maps}
    map_paths = [maps_by_name[path.stem + "_fixMap"] for path in image_paths]

    test_set = create_dataset(image_paths, map_paths)
    test_names = [path.name for path in image_paths]

else:
    image_paths = create_path(f"{DATASET}/images/")
    map_paths = create_path(f"{DATASET}/maps/")

    test_set = create_dataset(image_paths, map_paths)
    test_names = [path.name for path in image_paths]

test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)

example_indices = np.random.default_rng(42).choice(len(test_set), num_examples, replace=False).tolist()

print("Dataset:", DATASET)
print("Test samples:", len(test_set))

Dataset: MIT-dataset
Test samples: 1003


## Metrics

In [46]:
def compute_metrics(prediction, target):
    p = np.asarray(prediction, dtype=np.float64)
    q = np.asarray(target, dtype=np.float64)
    #CC
    cc = np.corrcoef(p.flatten(), q.flatten())[0, 1] if p.std() > 0 and q.std() > 0 else np.nan
    #SIM and KLdivergence
    sim, kl = np.nan, np.nan
    if p.sum() > 0 and q.sum() > 0:
        p, q = p / p.sum(), q / q.sum()
        sim = np.minimum(p, q).sum()
        p, q = p + 1e-12, q + 1e-12
        p, q = p / p.sum(), q / q.sum()
        kl = max(0., float(np.sum(q * np.log(q / p))))
    return float(cc), float(sim), float(kl)


def load_model(architecture, path):
    model = SSResNet18() if architecture == 'SS' else MLResNet18()
    checkpoint = torch.load(path, map_location='cpu', weights_only=True)
    if 'model_state_dict' in checkpoint:
        checkpoint = checkpoint['model_state_dict']
    elif 'state_dict' in checkpoint:
        checkpoint = checkpoint['state_dict']
    model.load_state_dict(checkpoint, strict=True)
    return model.to(device).eval()

## 3-fold testing

In [50]:
results = {}
examples = {}

for name, (architecture, checkpoint_path) in checkpoints.items():
    model = load_model(architecture, checkpoint_path)
    values = []
    examples[name] = {}

    sample_index = 0

    with torch.inference_mode():
        for images, maps in test_loader:
            output = model(images.to(device))
            logits = output["fused"] if isinstance(output, dict) else output
            predictions = torch.sigmoid(logits).cpu().numpy()

            for j in range(len(images)):
                i = sample_index + j

                prediction = predictions[j, 0]
                target = maps[j, 0].numpy()

                cc, sim, kl = compute_metrics(prediction, target)
                values.append([test_names[i], cc, sim, kl])

                if i in example_indices:
                    examples[name][i] = prediction

            sample_index += len(images)

    # metric for each image
    with open(output_dir / f"{name}_metrics.csv", "w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(["image", "CC", "SIM", "KL"])
        writer.writerows(values)

    # mean metric for current fold
    scores = np.array([row[1:] for row in values])
    means = scores.mean(axis=0)
    results[name] = means.tolist()
    print(f"{name}: CC={means[0]:.4f}, SIM={means[1]:.4f}, KL={means[2]:.4f}")

    del model

    if device.type == "cuda":
        torch.cuda.empty_cache()

with open(output_dir / "summary.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["model", "CC", "SIM", "KL"])
    writer.writerows([name] + values for name, values in results.items())

# mean and standard deviation across folds: computed on the results of the three folds, not the testing of the ensemble 
def summarize_folds(results, runs, folds):
    summary = {}

    for run in runs:
        values = np.array([results[f"{run}_fold{fold}"] for fold in folds])
        summary[run] = (values.mean(axis=0), values.std(axis=0, ddof=1))
    return summary

fold_summary = summarize_folds(results, RUNS, folds)

with open(output_dir / "summary_folds.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["model", "n_folds", "CC_mean", "CC_std", "SIM_mean", "SIM_std", "KL_mean", "KL_std"])

    for run, (means, stds) in fold_summary.items():
        writer.writerow([run, len(folds), means[0], stds[0], means[1], stds[1], means[2], stds[2]])

        print(f"{run}: CC={means[0]:.4f} ± {stds[0]:.4f}, SIM={means[1]:.4f} ± {stds[1]:.4f}, KL={means[2]:.4f} ± {stds[2]:.4f}")

KeyboardInterrupt: 